# Clean the Star Ratings quarterly data and insert it into DB

This notebook builds a data pipeline for the Star Ratings quarterly extract and writes two tables:

- `star_ratings_raw` — faithful 78-column mirror of the *Detailed data* sheet
- `star_ratings` — curated table for the frontend, with the four-level RE
  responses collapsed into a single 1.0–5.0 score per topic

## Cleaning rules
- read the *Detailed data* sheet (78 columns)
- normalize whitespace in text fields, blank → missing
- coerce ratings to integers (1–5, NULL allowed)
- coerce care minutes and QM rates to numeric, NULL allowed
- parse `[C] Date Decision Applied/Ends` from `d/m/yyyy` strings to real dates
- preserve NULLs throughout — do not fill rating/percentage gaps with 0

## Linking to `aged_care_services`
- soft link via `service_id`, nullable, no FK constraint
- match on `(service_name, suburb)` against residential services only
- only assign `service_id` when there is exactly one match — ambiguous keys get NULL
- ~98% of star ratings rows are expected to resolve

## RE topic score (1.0–5.0)
Weighted average of the four answer levels with evenly-spaced weights:

| Level             | Weight |
|-------------------|--------|
| Always            | 5      |
| Most of the time  | 11/3   |
| Some of the time  | 7/3    |
| Never             | 1      |

Score = Σ(pct × weight) / Σ(pct), preserving NULL when no survey data exists.


In [1]:
from pathlib import Path
import re
import pandas as pd

INPUT_XLSX = Path("input/star-ratings-quarterly-data-extract-february-2026.xlsx")

INPUT_XLSX

PosixPath('input/star-ratings-quarterly-data-extract-february-2026.xlsx')

In [2]:
df = pd.read_excel(INPUT_XLSX, sheet_name="Detailed data")
df.shape, df.columns[:10].tolist()


((2582, 78),
 ['Reporting Period',
  'Service Name',
  'Provider Name',
  'Service Suburb',
  'Purpose',
  'Aged Care Planning Region',
  'State/Territory',
  'MMM Region',
  'MMM Code',
  'Size'])

In [3]:
RE_TOPICS = ["Food", "Safety", "Operation", "Care Need", "Competent",
             "Independent", "Explain", "Respect", "Follow Up",
             "Caring", "Voice", "Home"]
LEVELS = ["Always", "Most of the time", "Some of the time", "Never"]


def _topic_slug(topic: str) -> str:
    return topic.lower().replace(" ", "_")


def _level_slug(level: str) -> str:
    return level.lower().replace(" ", "_")


def _build_column_mapping():
    m = {
        "Reporting Period": "reporting_period",
        "Service Name": "service_name",
        "Provider Name": "provider_name",
        "Service Suburb": "service_suburb",
        "Purpose": "purpose",
        "Aged Care Planning Region": "aged_care_planning_region",
        "State/Territory": "state_territory",
        "MMM Region": "mmm_region",
        "MMM Code": "mmm_code",
        "Size": "size",
        "Overall Star Rating": "overall_star_rating",
        "Residents’ Experience rating": "residents_experience_rating",
        "Residents\' Experience rating": "residents_experience_rating",
        "[RE] Interview Year": "re_interview_year",
        "Compliance rating": "compliance_rating",
        "[C] Decision type": "c_decision_type",
        "[C] Date Decision Applied": "c_date_decision_applied",
        "[C] Date Decision Ends": "c_date_decision_ends",
        "Staffing rating": "staffing_rating",
        "[S] Registered Nurse Care Minutes - Target": "s_rn_care_minutes_target",
        "[S] Registered Nurse Care Minutes - Actual": "s_rn_care_minutes_actual",
        "[S] Total Care Minutes - Target": "s_total_care_minutes_target",
        "[S] Total Care Minutes - Actual": "s_total_care_minutes_actual",
        "Quality Measures rating": "quality_measures_rating",
        "[QM] Pressure injuries*": "qm_pressure_injuries",
        "[QM] Restrictive practices": "qm_restrictive_practices",
        "[QM] Unplanned weight loss*": "qm_unplanned_weight_loss",
        "[QM] Falls and major injury - falls*": "qm_falls",
        "[QM] Falls and major injury - major injury from a fall*": "qm_falls_major_injury",
        "[QM] Medication management - polypharmacy": "qm_medication_polypharmacy",
        "[QM] Medication management - antipsychotic": "qm_medication_antipsychotic",
    }
    for topic in RE_TOPICS:
        for level in LEVELS:
            src = f"[RE] {topic} - {level}"
            dst = f"re_{_topic_slug(topic)}_{_level_slug(level)}"
            m[src] = dst
    return m


COLUMN_MAPPING = _build_column_mapping()


def clean_star_ratings(input_path: Path):
    df = pd.read_excel(input_path, sheet_name="Detailed data")

    df = df.rename(columns=COLUMN_MAPPING)
    df = df.dropna(axis=0, how="all").dropna(axis=1, how="all")

    # Normalize whitespace in text columns; blanks -> NA.
    text_cols = [
        "reporting_period", "service_name", "provider_name", "service_suburb",
        "purpose", "aged_care_planning_region", "state_territory",
        "mmm_region", "mmm_code", "size", "re_interview_year",
        "c_decision_type",
    ]
    for col in text_cols:
        if col in df.columns:
            s = df[col].astype("string")
            s = s.str.replace(r"\s+", " ", regex=True).str.strip()
            s = s.replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
            df[col] = s

    # Star ratings: 1-5, nullable. Use Int64 so NaN is preserved.
    rating_cols = [
        "overall_star_rating", "residents_experience_rating",
        "compliance_rating", "staffing_rating", "quality_measures_rating",
    ]
    for col in rating_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")

    # RE answer percentages: numeric, NaN preserved (no fillna).
    for topic in RE_TOPICS:
        for level in LEVELS:
            col = f"re_{_topic_slug(topic)}_{_level_slug(level)}"
            if col in df.columns:
                df[col] = pd.to_numeric(df[col], errors="coerce").round(2)

    # Compliance dates: source is dd/mm/yyyy strings.
    for col in ["c_date_decision_applied", "c_date_decision_ends"]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], dayfirst=True, errors="coerce").dt.date

    # Staffing care minutes and QM rates: numeric, NaN preserved.
    numeric_cols = [
        "s_rn_care_minutes_target", "s_rn_care_minutes_actual",
        "s_total_care_minutes_target", "s_total_care_minutes_actual",
        "qm_pressure_injuries", "qm_restrictive_practices",
        "qm_unplanned_weight_loss", "qm_falls", "qm_falls_major_injury",
        "qm_medication_polypharmacy", "qm_medication_antipsychotic",
    ]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").round(2)

    rows_before = len(df)
    df = df.drop_duplicates().reset_index(drop=True)

    summary = {
        "input_rows": rows_before,
        "output_rows": len(df),
        "duplicates_removed": rows_before - len(df),
        "column_count": df.shape[1],
    }
    return df, summary


In [4]:
raw_df, summary = clean_star_ratings(INPUT_XLSX)
summary


{'input_rows': 2582,
 'output_rows': 2582,
 'duplicates_removed': 0,
 'column_count': 78}

In [5]:
raw_df.head()


,reporting_period,service_name,provider_name,service_suburb,purpose,aged_care_planning_region,state_territory,mmm_region,mmm_code,size,...,s_total_care_minutes_target,s_total_care_minutes_actual,quality_measures_rating,qm_pressure_injuries,qm_restrictive_practices,qm_unplanned_weight_loss,qm_falls,qm_falls_major_injury,qm_medication_polypharmacy,qm_medication_antipsychotic
0,February 2026,Anglicare Mildred Symons House,Anglican Community Services,JANNALI,Not for Profit,South East Sydney,NSW,Metropolitan,MM1,Large,...,228.0,229.0,5,0.0,1.0,2.0,26.0,3.0,32.0,3.0
1,February 2026,Bowral House Nursing Home,Thompson Health Care Pty Ltd,BOWRAL,For profit,South West Sydney,NSW,Large rural towns,MM3,Medium,...,212.0,218.0,3,2.0,22.0,8.0,31.0,0.0,31.0,13.0
2,February 2026,Whiddon Moree,The Frank Whiddon Masonic Homes of New South W...,MOREE,Not for Profit,New England,NSW,Medium rural towns,MM4,Small,...,231.0,237.0,3,0.0,26.0,9.0,39.0,0.0,31.0,17.0
3,February 2026,William Beech Gardens - Bushmaster,RSL LifeCare Limited,CONDOBOLIN,Not for Profit,Central West,NSW,Small rural towns,MM5,Small,...,236.0,236.0,2,2.0,35.0,8.0,52.0,2.0,37.0,21.0
4,February 2026,Uniting Wesley Heights Manly,The Uniting Church in Australia Property Trust...,MANLY,Not for Profit,Northern Sydney,NSW,Metropolitan,MM1,Medium,...,215.0,216.0,5,0.0,0.0,3.0,40.0,3.0,24.0,0.0


In [6]:
print(f"Reporting periods: {raw_df['reporting_period'].unique().tolist()}")
print(f"Unique services: {raw_df.groupby(['service_name','provider_name']).ngroups}")
print(f"Rows missing all 5 ratings: {raw_df[['overall_star_rating','residents_experience_rating','compliance_rating','staffing_rating','quality_measures_rating']].isna().all(axis=1).sum()}")
print(f"Rows with any RE survey data: {raw_df['re_food_always'].notna().sum()}")


Reporting periods: ['February 2026']
Unique services: 2577
Rows missing all 5 ratings: 153
Rows with any RE survey data: 2416


# Transform: build the curated dataframe

Compute the 12 `re_<topic>_score` columns on a 1.0–5.0 scale and project
the columns the frontend needs.


In [7]:
WEIGHTS = {
    "always": 5.0,
    "most_of_the_time": 11.0 / 3.0,
    "some_of_the_time": 7.0 / 3.0,
    "never": 1.0,
}


def compute_topic_score(df: pd.DataFrame, topic: str) -> pd.Series:
    """Weighted-average score for one RE topic, on a 1.0-5.0 scale.

    Returns NaN when the service has no survey data for that topic.
    Divides by the sum of percentages (not by 100) so rounding error in
    the source data does not bias the score.
    """
    slug = _topic_slug(topic)
    cols = {level: df[f"re_{slug}_{level}"] for level in WEIGHTS}

    # NaN-aware: a column with NaN contributes 0 to both numerator and denominator
    total = sum(c.fillna(0) for c in cols.values())
    weighted = sum(c.fillna(0) * w for c, w in zip(cols.values(), WEIGHTS.values()))

    score = weighted / total
    # Preserve NaN where the service had no survey data at all
    return score.where(total > 0).round(2)


def transform_to_curated(raw_df: pd.DataFrame) -> pd.DataFrame:
    base_cols = [
        "service_id",
        "reporting_period", "service_name", "provider_name",
        "service_suburb", "state_territory",
        "overall_star_rating", "residents_experience_rating",
        "compliance_rating", "staffing_rating", "quality_measures_rating",
        "s_rn_care_minutes_target", "s_rn_care_minutes_actual",
        "s_total_care_minutes_target", "s_total_care_minutes_actual",
    ]
    curated = raw_df[[c for c in base_cols if c in raw_df.columns]].copy()

    for topic in RE_TOPICS:
        curated[f"re_{_topic_slug(topic)}_score"] = compute_topic_score(raw_df, topic)

    return curated


# Insert into DB

Connect, create both tables from their schema files, resolve `service_id`
against `aged_care_services`, then insert raw and curated.


In [8]:
import os
from sqlalchemy import create_engine, text
from dotenv import load_dotenv

# -----------------------------
# 1. DATABASE CONNECTION
# -----------------------------
load_dotenv(".env.db", override=True)

DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")

DATABASE_URL = (
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}"
    f"@{DB_HOST}:{DB_PORT}/{DB_NAME}?sslmode=require"
)

engine = create_engine(DATABASE_URL)

# -----------------------------
# 2. CREATE TABLES FROM SCHEMA
# -----------------------------
for schema_file in ["schemas/star_ratings_raw.sql", "schemas/star_ratings.sql"]:
    with open(schema_file, "r", encoding="utf-8") as f:
        schema_sql = f.read()
    with engine.begin() as conn:
        conn.execute(text(schema_sql))
    print(f"Applied {schema_file}")

print(DB_NAME)


Applied schemas/star_ratings_raw.sql
Applied schemas/star_ratings.sql
carelink


In [9]:
# -----------------------------
# 3. RESOLVE service_id
# -----------------------------
# Soft link: match on (service_name, service_suburb) against
# residential rows in aged_care_services. Only assign service_id when
# there is exactly one match - ambiguous keys (e.g. Mirinjani) and
# missing services get NULL.

services = pd.read_sql(
    """
    SELECT id AS service_id, service_name, physical_suburb AS service_suburb
    FROM aged_care_services
    WHERE care_type = 'Residential'
    """,
    engine,
)


def _norm(s):
    return s.astype("string").str.replace(r"\s+", " ", regex=True).str.strip().str.upper()


services["_svc"] = _norm(services["service_name"])
services["_sub"] = _norm(services["service_suburb"])

# Drop ambiguous keys (multiple matches) before merging.
counts = services.groupby(["_svc", "_sub"]).size()
unique_keys = counts[counts == 1].index
services_unique = services.set_index(["_svc", "_sub"]).loc[unique_keys].reset_index()
print(f"Residential services available for linking: {len(services)}")
print(f"  Unambiguous (svc, suburb) keys: {len(services_unique)}")
print(f"  Ambiguous keys excluded:        {len(services) - len(services_unique)}")

raw_df["_svc"] = _norm(raw_df["service_name"])
raw_df["_sub"] = _norm(raw_df["service_suburb"])

raw_df = raw_df.merge(
    services_unique[["service_id", "_svc", "_sub"]],
    on=["_svc", "_sub"],
    how="left",
)
raw_df = raw_df.drop(columns=["_svc", "_sub"])

unmatched = raw_df["service_id"].isna().sum()
print(f"\nResolved: {len(raw_df) - unmatched}/{len(raw_df)} "
      f"({(1 - unmatched/len(raw_df))*100:.1f}%)")
print(f"Unmatched (will insert with service_id = NULL): {unmatched}")


Residential services available for linking: 2590
  Unambiguous (svc, suburb) keys: 2580
  Ambiguous keys excluded:        10

Resolved: 2523/2582 (97.7%)
Unmatched (will insert with service_id = NULL): 59


In [10]:
curated_df = transform_to_curated(raw_df)
curated_df.head()

,service_id,reporting_period,service_name,provider_name,service_suburb,state_territory,overall_star_rating,residents_experience_rating,compliance_rating,staffing_rating,...,re_operation_score,re_care_need_score,re_competent_score,re_independent_score,re_explain_score,re_respect_score,re_follow_up_score,re_caring_score,re_voice_score,re_home_score
0,16ad5d18-3df7-42d5-80cc-c8fe4dec10ed,February 2026,Anglicare Mildred Symons House,Anglican Community Services,JANNALI,NSW,4,3,5,3,...,3.79,4.62,3.88,4.45,3.85,4.33,4.11,4.33,4.67,4.18
1,45d37607-d2eb-4bc4-94cb-4433b97f3368,February 2026,Bowral House Nursing Home,Thompson Health Care Pty Ltd,BOWRAL,NSW,4,4,5,4,...,4.43,4.81,4.24,4.43,4.05,4.43,4.63,4.52,4.81,4.81
2,ddcc5070-b271-4e71-855b-4f3e404f3761,February 2026,Whiddon Moree,The Frank Whiddon Masonic Homes of New South W...,MOREE,NSW,3,3,4,3,...,3.88,4.29,3.76,4.08,3.46,4.39,3.87,3.87,4.37,3.57
3,345f630f-dfec-4db2-9049-0904912db81e,February 2026,William Beech Gardens - Bushmaster,RSL LifeCare Limited,CONDOBOLIN,NSW,3,3,4,4,...,3.97,3.98,4.07,3.87,3.67,4.17,3.46,4.59,3.87,3.36
4,bd60845b-5c33-44f1-851b-ed135e8fe490,February 2026,Uniting Wesley Heights Manly,The Uniting Church in Australia Property Trust...,MANLY,NSW,4,4,5,3,...,4.20,4.67,4.53,4.73,4.13,4.47,4.60,4.60,4.87,4.40


In [11]:
# -----------------------------
# 4. INSERT INTO star_ratings_raw
# -----------------------------
raw_target_cols = [
    "service_id",
    "reporting_period", "service_name", "provider_name", "service_suburb",
    "purpose", "aged_care_planning_region", "state_territory",
    "mmm_region", "mmm_code", "size",
    "overall_star_rating", "residents_experience_rating",
    "compliance_rating", "staffing_rating", "quality_measures_rating",
    "re_interview_year",
]
for topic in RE_TOPICS:
    for level in LEVELS:
        raw_target_cols.append(f"re_{_topic_slug(topic)}_{_level_slug(level)}")
raw_target_cols += [
    "c_decision_type", "c_date_decision_applied", "c_date_decision_ends",
    "s_rn_care_minutes_target", "s_rn_care_minutes_actual",
    "s_total_care_minutes_target", "s_total_care_minutes_actual",
    "qm_pressure_injuries", "qm_restrictive_practices",
    "qm_unplanned_weight_loss", "qm_falls", "qm_falls_major_injury",
    "qm_medication_polypharmacy", "qm_medication_antipsychotic",
]

raw_insert = raw_df[raw_target_cols].copy()

raw_insert.to_sql(
    "star_ratings_raw",
    con=engine,
    if_exists="append",
    index=False,
    method="multi",
    chunksize=500,
)

print(f"Inserted {len(raw_insert)} rows into star_ratings_raw")

Inserted 2582 rows into star_ratings_raw


In [12]:
# -----------------------------
# 5. INSERT INTO star_ratings
# -----------------------------
curated_target_cols = [
    "service_id",
    "reporting_period", "service_name", "provider_name",
    "service_suburb", "state_territory",
    "overall_star_rating", "residents_experience_rating",
    "compliance_rating", "staffing_rating", "quality_measures_rating",
]
for topic in RE_TOPICS:
    curated_target_cols.append(f"re_{_topic_slug(topic)}_score")
curated_target_cols += [
    "s_rn_care_minutes_target", "s_rn_care_minutes_actual",
    "s_total_care_minutes_target", "s_total_care_minutes_actual",
]

curated_insert = curated_df[curated_target_cols].copy()

curated_insert.to_sql(
    "star_ratings",
    con=engine,
    if_exists="append",
    index=False,
    method="multi",
    chunksize=1000,
)

print(f"Inserted {len(curated_insert)} rows into star_ratings")


Inserted 2582 rows into star_ratings


In [13]:
# -----------------------------
# 6. VALIDATE
# -----------------------------
with engine.begin() as conn:
    raw_count = conn.execute(text("SELECT COUNT(*) FROM star_ratings_raw")).scalar()
    cur_count = conn.execute(text("SELECT COUNT(*) FROM star_ratings")).scalar()
    raw_linked = conn.execute(text(
        "SELECT COUNT(*) FROM star_ratings_raw WHERE service_id IS NOT NULL"
    )).scalar()
    cur_linked = conn.execute(text(
        "SELECT COUNT(*) FROM star_ratings WHERE service_id IS NOT NULL"
    )).scalar()

print(f"star_ratings_raw: {raw_count} rows ({raw_linked} linked)")
print(f"star_ratings:     {cur_count} rows ({cur_linked} linked)")


star_ratings_raw: 2582 rows (2523 linked)
star_ratings:     2582 rows (2523 linked)
